# Stochastic Modeling of Surgical Planning (SAA Method)

In practice, the actual durations of surgical interventions are difficult to predict due to unforeseen operational events. To represent this variability, we introduce a stochastic model based on scenarios $\omega \in \Omega$. We use the **Sample Average Approximation (SAA)** method.

## 1. Introducing Uncertainty

For each patient $i$, the operating time is decomposed into two parameters:
* $t_{i,fixe}$: the minimal, deterministic operating time required for patient $i$.
* $t_{i,sup}$: the maximum additional time the patient's surgery might require in case of unexpected events.

In each scenario $\omega \in \Omega$, the actual duration of the operation is modeled by the random variable:
$$t_{i}^{\omega} = t_{i,fixe} + \alpha_{i}^{\omega}t_{i,sup}$$
where $\alpha_{i}^{\omega} \sim \mathcal{U}[0,1]$ represents the proportion of additional time realized. 

## 2. Two-Stage Decisions

The stochastic model is a two-stage linear program:
* **First Stage Decisions:** Decisions made *before* the uncertainty is realized, namely the assignment of patients ($x_{ik}$) and operating rooms ($y_{jk}$).
* **Second Stage (Recourse) Decisions:** The overtime hours ($\tau_{jk}^{\omega}$) calculated *after* the realization of scenario $\omega$.

## 3. SAA Formulation

When the expected value is approximated by a finite set of scenarios $\Omega = \{s_1, ..., s_S\}$ with probabilities $p_s = 1/S$, the model becomes a single large linear program.

### Objective Function
The objective is to minimize the total cost, composed of the certain first-stage cost and the expected second-stage cost:

$$\min Z_{SAA} = \sum_{j \in J}\sum_{k \in K}C_{jk}y_{jk} + \sum_{s=1}^{S}p_{s}\left(\sum_{j \in J}\sum_{k \in K}C_{jk}^{\prime}\tau_{jk}^{s}\right)$$

### Modified Constraints
The structural constraints (unique patient assignment, daily capacity, and specialty consistency) remain identical to the deterministic case because $x_{ik}$ and $y_{jk}$ do not depend on the scenario. 

However, the overtime constraint must be satisfied for *each* scenario $s$:

$$\sum_{i | s_i = j}(t_{i,fixe} + \alpha_{i}^{s}t_{i,sup})x_{ik} - H y_{jk} \le \tau_{jk}^{s} \quad \forall j, \forall k, \forall s$$

With the bounds:
$$0 \le \tau_{jk}^{s} \le H_{max\_sup} y_{jk} \quad \forall j, \forall k, \forall s$$

In [ ]:
import pulp
import pandas as pd
import numpy as np

### 1. Data Loading and Harmonization
Just like the deterministic model, we start by loading our patient and specialty data. We also import `numpy` to handle the random generation of scenarios.

In [ ]:
# Excel file name
FILENAME = "data/surgery_data.xlsx"

try:
    # Load the first sheet (patients)
    df = pd.read_excel(FILENAME, sheet_name="Patients")
    df.columns = ['Patient_ID', 'Spec_ID', 'Specialty', 'Duration', 'Overtime']

    # Load the second sheet (costs)
    df_costs = pd.read_excel(FILENAME, sheet_name="Specialties")

except FileNotFoundError:
    print(f"ERROR: The file '{FILENAME}' was not found.")
    exit()

# --- Name harmonization to avoid inconsistencies ---
df['Specialty'] = df['Specialty'].str.strip().str.upper()
df_costs['names'] = df_costs['names'].str.strip().str.upper()

# Sets definition
patients = [f"P{i}" for i in df['Patient_ID']]
specialties = df['Specialty'].unique().tolist()
days = list(range(1, 8))  # 1=Monday, ..., 7=Sunday
day_map = {1: "Monday", 2: "Tuesday", 3: "Wednesday", 4: "Thursday", 5: "Friday", 6: "Saturday", 7: "Sunday"}

# --- Cost extraction from the "Specialties" sheet ---
base_fixed_costs = {}
base_overtime_costs = {}

for _, row in df_costs.iterrows():
    base_fixed_costs[row['names']] = row['fixed cost']
    base_overtime_costs[row['names']] = row['overtime cost (per h)']

# Daily Costs Calculation (with weekend multipliers)
C_jk = {}
C_prime_jk = {}
for j in specialties:
    for k in days:
        multiplier = 1.25 if k == 6 else (1.8 if k == 7 else 1.0)
        C_jk[(j, k)] = base_fixed_costs[j] * multiplier
        C_prime_jk[(j, k)] = base_overtime_costs[j] * multiplier

# Time constraints
H = 8.0         # Standard daily working hours
H_max_sup = 2.0 # Maximum allowed overtime hours

### 2. SAA Phase 1: Scenario Generation for Optimization
In this step, we generate a small set of training scenarios ($S = 10$). For each patient in each scenario, the actual surgery duration is calculated using a random factor $\alpha \sim \mathcal{U}[0,1]$ applied to their maximum potential overtime.

In [2]:
# --- Scenario Generation for SAA Optimization ---
S = 10  # Number of training scenarios for optimization
scenarios = list(range(1, S + 1))
prob_s = 1.0 / S  # Equal probability for each scenario

# patient_specialty dictionary
patient_specialty = {f"P{row.Patient_ID}": row.Specialty for _, row in df.iterrows()}

# durations_s[s][i] = randomized duration of patient i in scenario s
durations_s = {s: {} for s in scenarios}

np.random.seed(42)  # For reproducibility

for s in scenarios:
    for _, row in df.iterrows():
        patient_id = f"P{row.Patient_ID}"
        alpha = np.random.uniform(0, 1)  # Random delay between 0% and 100% of max overtime
        actual_duration = row.Duration + alpha * row.Overtime
        durations_s[s][patient_id] = actual_duration

print(f"Generated {S} training scenarios for the Sample Average Approximation (SAA).")

Generated 10 training scenarios for the Sample Average Approximation (SAA).



### 3. First-Stage and Second-Stage Variables
Here we define the core of our stochastic model. 
* **First-Stage Variables ($X, Y$):** Patient assignments and room openings. These are decided *now* and do not depend on the scenarios.
* **Second-Stage Variables ($\tau$):** Overtime hours. These adjust automatically based on the scenario $s$.

In [3]:
# --- Modeling with PuLP ---
model = pulp.LpProblem("Stochastic_Surgical_Planning", pulp.LpMinimize)

# 1st Stage Variables (Independent of scenarios)
x = pulp.LpVariable.dicts("X", (patients, days), cat='Binary')
y = pulp.LpVariable.dicts("Y", (specialties, days), cat='Binary')

# 2nd Stage Variables (Dependent on scenarios)
tau = pulp.LpVariable.dicts("Tau", (specialties, days, scenarios), lowBound=0)

# Objective Function: Fixed costs + Expected (Average) Overtime costs across scenarios
fixed_cost_expr = pulp.lpSum(C_jk[(j, k)] * y[j][k] for j in specialties for k in days)
expected_overtime_cost_expr = pulp.lpSum(
    prob_s * C_prime_jk[(j, k)] * tau[j][k][s]
    for j in specialties for k in days for s in scenarios
)
model += fixed_cost_expr + expected_overtime_cost_expr, "Expected_Total_Cost"


### 4. Constraints
The structural constraints remain the same. However, the overtime calculation and limitations must be enforced for **every single scenario** to ensure the schedule works no matter what happens in our training set.

In [4]:
# --- Constraints ---

# 1. Each patient is assigned to exactly one day
for i in patients:
    model += pulp.lpSum(x[i][k] for k in days) == 1

# 2. Only one specialty per day
for k in days:
    model += pulp.lpSum(y[j][k] for j in specialties) <= 1

# 3. Consistency: Patient can only be assigned if their specialty is scheduled
for i in patients:
    for k in days: 
        model += x[i][k] <= y[patient_specialty[i]][k]

# 4. Overtime calculation (per scenario)
for j in specialties:
    for k in days:
        for s in scenarios:
            total_duration_s = pulp.lpSum(durations_s[s][i] * x[i][k] for i in patients if patient_specialty[i] == j)
            model += total_duration_s - H * y[j][k] <= tau[j][k][s]

# 5. Overtime limitation (per scenario)
for j in specialties:
    for k in days:
        for s in scenarios:
            model += tau[j][k][s] <= H_max_sup * y[j][k]

### 5. Solving the Master Problem
We solve the model to find the optimal *First-Stage* schedule. Once found, this schedule is "frozen" (fixed) for the next phase.


In [5]:
print("\nSolving SAA Model in progress...")
model.solve(pulp.GUROBI(msg=0))
# If Gurobi is not installed, use: model.solve(pulp.PULP_CBC_CMD(msg=0))

if pulp.LpStatus[model.status] == 'Optimal':
    print(f"Status: Optimal")
    print(f"Optimized Expected Total Cost: {pulp.value(model.objective):.2f} €")
    
    # Extract the "frozen" First-Stage schedule
    planned_x = {i: {k: x[i][k].varValue for k in days} for i in patients}
    planned_y = {j: {k: y[j][k].varValue for k in days} for j in specialties}
else:
    print("No optimal solution found. The model might be infeasible.")
    exit()


Solving SAA Model in progress...
Restricted license - for non-production use only - expires 2027-11-29
Status: Optimal
Optimized Expected Total Cost: 18033.61 €


### 6. Phase 2: Stress-Testing the Schedule
Now that we have our fixed schedule, we need to see how it performs in the real world. We generate 100 new, extreme scenarios (delays up to 150%) and simulate our schedule against them to evaluate its true cost and feasibility. 

*Note: We stress-test with delays up to 150% because testing at 100% yields a 100% reliability rate. Pushing the boundaries allows us to properly observe the model's limits and robustness under truly extreme conditions.*

In [9]:
# --- Phase 2: Evaluation on N Test Scenarios ---
N_TEST = 100
test_scenarios = list(range(1, N_TEST + 1))
ALPHA_MAX_TEST = 1.5  # Extreme stress test: up to 150% of expected overtime

np.random.seed(100) # Different seed for test set
test_costs = []
feasible_scenarios = 0

for s in test_scenarios:
    scenario_cost = 0.0
    is_feasible = True
    
    # 1. Fixed costs (independent of scenario)
    for j in specialties:
        for k in days:
            if planned_y[j][k] > 0.5:
                scenario_cost += C_jk[(j, k)]
                
    # 2. Overtime calculation for this specific test scenario
    for k in days:
        for j in specialties:
            if planned_y[j][k] > 0.5:
                # Calculate real time consumed by scheduled patients today
                real_time_today = 0.0
                for i in patients:
                    if planned_x[i][k] > 0.5 and patient_specialty[i] == j:
                        # Extract data from dataframe
                        base_dur = df[df['Patient_ID'] == int(i[1:])]['Duration'].values[0]
                        max_over = df[df['Patient_ID'] == int(i[1:])]['Overtime'].values[0]
                        
                        # Apply extreme test alpha
                        alpha_test = np.random.uniform(0, ALPHA_MAX_TEST)
                        real_time_today += base_dur + alpha_test * max_over
                
                # Check limits
                if real_time_today > H:
                    overtime_needed = real_time_today - H
                    scenario_cost += overtime_needed * C_prime_jk[(j, k)]
                    
                    # If total time exceeds absolute physical limit (10h), scenario fails
                    if real_time_today > H + H_max_sup:
                        is_feasible = False
                        
    test_costs.append(scenario_cost)
    if is_feasible:
        feasible_scenarios += 1

print("\n--- STRESS TEST RESULTS (100 Extreme Scenarios) ---")
print(f"Severity Factor (Alpha Max) : {ALPHA_MAX_TEST}")
print(f"Average Realized Cost       : {np.mean(test_costs):.2f} €")
print(f"Cost Standard Deviation     : {np.std(test_costs):.2f} €")
print(f"Reliability (Feasibility)   : {feasible_scenarios / N_TEST * 100:.1f} %")


--- STRESS TEST RESULTS (100 Extreme Scenarios) ---
Severity Factor (Alpha Max) : 1.5
Average Realized Cost       : 20225.03 €
Cost Standard Deviation     : 832.57 €
Reliability (Feasibility)   : 75.0 %


### 7. Aggregated Analysis by Specialty
Finally, let's analyze how the stochastic model distributed the workload among the different surgical specialties. This shows us the theoretical "fill rate" of the operating rooms.


In [10]:
# --- Aggregated Analysis ---
print("\n--- AGGREGATED ANALYSIS BY SPECIALTY ---")

specialty_stats = []
for j in specialties:
    days_used = sum(1 for k in days if planned_y[j][k] > 0.5)
    if days_used > 0:
        assigned_patients = [i for i in patients for k in days if planned_x[i][k] > 0.5 and patient_specialty[i] == j]
        
        # Calculate theoretical nominal time
        nominal_time = sum(df[df['Patient_ID'] == int(i[1:])]['Duration'].values[0] for i in assigned_patients)
        fill_rate = (nominal_time / (days_used * H)) * 100
        
        specialty_stats.append({
            "Specialty": j,
            "Days Allocated": days_used,
            "Patients Handled": len(assigned_patients),
            "Nominal Time (h)": round(nominal_time, 2),
            "Capacity Fill Rate (%)": round(fill_rate, 1)
        })

df_stats = pd.DataFrame(specialty_stats)
df_stats


--- AGGREGATED ANALYSIS BY SPECIALTY ---


,Specialty,Days Allocated,Patients Handled,Nominal Time (h),Capacity Fill Rate (%)
0,ORTHO,2,17,13.90,86.9
1,GEN,1,7,6.10,76.2
2,ENT,2,17,9.54,59.6
3,OBGYN,1,10,5.12,64.0
4,VASCULAR,1,5,7.43,92.9
